# AI-Publisher Docker Image Builder v2

Bu notebook, AI-Publisher platformu icin Docker imajlarini Google Colab'da Kaniko ile build eder.
Imajlar Google Drive'a `.tar.gz` olarak kaydedilir, yerel makinede `docker compose up` ile yuklenir.

### Degisiklikler (v2):
- **Video codec fix**: cogvideox, wan, ltx, hunyuan, svd, animatediff, wan25 -> MJPG -> libx264 + yuv420p
- **Colab runtime kalkti**: Bu notebook sadece build icin, runtime sunucu Docker container'larda
- **Force-Rebuild**: Video modelleri zorunlu rebuild edilir (codec fix icin)
- **Incremental**: Ses/efekt modelleri Drive'da varsa atlanir

---
Her hucreyi sirayla calistirin.

## Hucre 1: Runtime Kontrolu

Sistem durumunu kontrol et. Build tamamen CPU calistigi icin GPU runtime gerekmez.
Runtime > Change runtime type > None secin.

In [ ]:
import os
_clean_env = os.environ.copy()
_clean_env.pop('LD_LIBRARY_PATH', None) # GLIBC hatasini onlemek icin
#@title Runtime Kontrolu { display-mode: "form" }
import subprocess
import os

def check_runtime():
    print("=" * 60)
    print("AI-Publisher Docker Build - Runtime Kontrolu")
    print("=" * 60)
    
    # GPU kontrolu
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=10
        , env=_clean_env)
        if result.returncode == 0:
            gpu_info = result.stdout.strip().split(", ")
            print(f"GPU: {gpu_info[0]}")
            print(f"   Toplam VRAM: {gpu_info[1]} MB")
            print(f"   Bos VRAM: {gpu_info[2]} MB")
        else:
            print("GPU bulunamadi. Build icin GPU gerekli degil, devam.")
    except FileNotFoundError:
        print("nvidia-smi bulunamadi. Build icin GPU gerekli degil, devam.")
    
    # Disk alani
    disk = subprocess.run(["df", "-h", "/content"], capture_output=True, text=True, env=_clean_env)
    if disk.returncode == 0:
        lines = disk.stdout.strip().split("\n")
        if len(lines) > 1:
            parts = lines[1].split()
            print(f"Disk: {parts[2]} kullanildi / {parts[1]} toplam")
    
    # RAM
    with open("/proc/meminfo") as f:
        for line in f:
            if line.startswith("MemTotal"):
                total_kb = int(line.split()[1])
                print(f"RAM: {total_kb // 1024} MB")
                break
    
    print("\nRuntime hazir. Sonraki hucreye gecin.")
    return True

check_runtime()

## Hucre 2: Drive Mount ve Bagimliliklar

Google Drive'i bagla, Kaniko + registry + pigz kur.

In [ ]:
import os
_clean_env = os.environ.copy()
_clean_env.pop('LD_LIBRARY_PATH', None) # GLIBC hatasini onlemek icin
#@title Drive Mount ve Bagimliliklar { display-mode: "form" }
import subprocess
import sys
import os
import time

print("Bagimliliklar kuruluyor...")

# pip kontrol
subprocess.run("rm -f /var/lib/dpkg/lock-frontend /var/lib/apt/lists/lock /var/cache/apt/archives/lock 2>/dev/null || true", shell=True, env=_clean_env)
subprocess.run("dpkg --configure -a 2>/dev/null || true", shell=True, env=_clean_env)
r = subprocess.run([sys.executable, "-m", "pip", "--version"], capture_output=True, text=True, env=_clean_env)
if r.returncode != 0:
    print("  pip bulunamadi, apt-get ile kuruluyor...")
    subprocess.run("apt-get update -q && apt-get install -y -q python3-pip", shell=True, check=True, timeout=120, env=_clean_env)

# Google Drive mount
try:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
        print("Google Drive baglandi.")
    else:
        print("Google Drive zaten bagli.")
except Exception as e:
    print(f"Drive mount hatasi: {e}")

print("\nKaniko + Registry + pigz kurulumu...")

# Kaniko kurulumu
kaniko_checks = [
    {"path": "/kaniko/executor", "label": "Kaniko"},
    {"path": "/usr/local/bin/registry", "label": "Registry"},
    {"path": "/usr/local/bin/pigz", "label": "pigz"},
]
all_ok = all(os.path.exists(c["path"]) for c in kaniko_checks)

if not all_ok:
    subprocess.run(["mkdir", "-p", "/kaniko"], check=True, env=_clean_env)
    subprocess.run(["apt-get", "update", "-qq"], check=True, timeout=120, env=_clean_env)
    
    # 1. Kaniko - GCR imajindan binary cikar (Docker gerekmez)
    print("  Kaniko GCR imajindan cikariliyor...")
    import requests, tarfile, io, stat
    
    kaniko_dest = '/kaniko/executor'
    if not os.path.exists(kaniko_dest):
        idx_resp = requests.get(
            'https://gcr.io/v2/kaniko-project/executor/manifests/latest',
            headers={'Accept': 'application/vnd.oci.image.index.v1+json'},
            timeout=30
        )
        idx_resp.raise_for_status()
        idx_data = idx_resp.json()
        
        amd64_digest = None
        for m in idx_data['manifests']:
            if m['platform']['architecture'] == 'amd64':
                amd64_digest = m['digest']
                break
        
        if not amd64_digest:
            raise RuntimeError('AMD64 manifest not found')
        
        manifest_resp = requests.get(
            f'https://gcr.io/v2/kaniko-project/executor/manifests/{amd64_digest}',
            headers={'Accept': 'application/vnd.oci.image.manifest.v1+json'},
            timeout=30
        )
        manifest_resp.raise_for_status()
        manifest_data = manifest_resp.json()
        
        found = False
        for i, layer in enumerate(manifest_data['layers']):
            digest = layer['digest']
            url = f'https://gcr.io/v2/kaniko-project/executor/blobs/{digest}'
            print(f'    Layer {i+1}/{len(manifest_data["layers"])} indiriliyor...')
            
            layer_resp = requests.get(url, timeout=120)
            layer_resp.raise_for_status()
            
            try:
                with tarfile.open(fileobj=io.BytesIO(layer_resp.content), mode='r:gz') as tar:
                    for member in tar.getmembers():
                        if member.name == 'kaniko/executor':
                            print(f'    kaniko/executor bulundu (layer {i+1})')
                            f = tar.extractfile(member)
                            if f:
                                with open(kaniko_dest, 'wb') as out:
                                    out.write(f.read())
                                st = os.stat(kaniko_dest)
                                os.chmod(kaniko_dest, st.st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
                                found = True
                                break
            except tarfile.ReadError:
                continue
            if found:
                break
        
        if not found:
            raise RuntimeError('kaniko/executor not found in GCR layers')
        
        if os.path.exists('/usr/local/bin/kaniko'):
            os.remove('/usr/local/bin/kaniko')
        os.symlink(kaniko_dest, '/usr/local/bin/kaniko')
        size_mb = os.path.getsize(kaniko_dest) / (1024 * 1024)
        print(f'    Kaniko kuruldu ({size_mb:.1f} MB)')
    else:
        print('    Kaniko zaten mevcut.')
    
    # 2. Registry - dist distribution binary
    print("  Registry indiriliyor...")
    if not os.path.exists('/usr/local/bin/registry'):
        import urllib.request
        url = 'https://github.com/distribution/distribution/releases/download/v2.8.3/registry_2.8.3_linux_amd64.tar.gz'
        try:
            urllib.request.urlretrieve(url, '/tmp/registry.tar.gz')
            import tarfile as tfgz
            with tfgz.open('/tmp/registry.tar.gz', 'r:gz') as tar:
                for member in tar.getmembers():
                    if member.name == 'registry':
                        f = tar.extractfile(member)
                        if f:
                            with open('/usr/local/bin/registry', 'wb') as out:
                                out.write(f.read())
                            os.chmod('/usr/local/bin/registry', 0o755)
                            print('    Registry kuruldu.')
                            break
            os.remove('/tmp/registry.tar.gz')
        except Exception as e:
            print(f'    WARN: Registry indirilemedi: {e}')
    else:
        print('    Registry zaten mevcut.')
    
    # 3. pigz
    print("  pigz kuruluyor...")
    subprocess.run(["apt-get", "install", "-y", "-qq", "pigz"], timeout=60, env=_clean_env)
    
    # 4. Registry config
    os.makedirs('/etc/docker/registry', exist_ok=True)
    os.makedirs('/var/lib/registry', exist_ok=True)
    with open('/etc/docker/registry/config.yml', 'w') as f:
        f.write('version: 0.1\n')
        f.write('log:\n')
        f.write('  level: error\n')
        f.write('storage:\n')
        f.write('  filesystem:\n')
        f.write('    rootdirectory: /var/lib/registry\n')
        f.write('http:\n')
        f.write('  addr: :5000\n')
    print("  Kaniko + Registry + pigz hazir.")
    print("Kaniko, Registry ve pigz basariyla kuruldu.")
else:
    print("Kaniko, Registry ve pigz zaten mevcut.")

print("\nDrive mount ve bagimliliklar tamam. Sonraki hucreye gecin.")

## Hucre 3: Repo Klonlama

GitHub'dan en son kodu cek. Codec fix'in dahil oldugundan emin olmak icin `main` branch'ini kullan.

In [ ]:
import os
_clean_env = os.environ.copy()
_clean_env.pop('LD_LIBRARY_PATH', None) # GLIBC hatasini onlemek icin
#@title Repo Klonlama { display-mode: "form" }
import subprocess
import os

REPO_URL = "https://github.com/Arda-Avci/AI-Publisher.git"
WORK_DIR = "/content/AI-Publisher"

print("Repo klonlaniyor...")

# GitHub PAT - sirasiyla dene: Colab secrets -> env var
GITHUB_PAT = ""

# 1) Colab secrets
try:
    from google.colab import userdata
    for name in ["GITHUB_PAT", "GITHUB_TOKEN", "GH_TOKEN"]:
        try:
            val = userdata.get(name)
            if val:
                GITHUB_PAT = val
                print(f"Colab secrets'dan '{name}' alindi.")
                break
        except Exception as e:
            print(f"  userdata.get('{name}') -> {e}")
except ImportError:
    print("google.colab.userdata mevcut degil (Colab disi ortam).")

# 2) Ortam degiskeni fallback
if not GITHUB_PAT:
    for name in ["GITHUB_PAT", "GITHUB_TOKEN", "GH_TOKEN"]:
        val = os.environ.get(name)
        if val:
            GITHUB_PAT = val
            print(f"Ortam degiskeni '{name}' alindi.")
            break

if not GITHUB_PAT:
    print("\n" + "="*60)
    print("HICBIR GITHUB TOKEN BULUNAMADI!")
    print("="*60)
    print("Colab Secrets'a ekleyin (SOLDAN: Anahtar simgesi -> Add new secret):")
    print("  Name: GITHUB_PAT")
    print("  Value: <github_pat>")
    print("  Notebook access: ON")
    print("\nAlternatif: Asagidaki forma token'i yapistirip calistirabilirsiniz.")
    try:
        from IPython.display import display
        import ipywidgets as widgets
        pat_input = widgets.Password(description="GITHUB_PAT:", style={"description_width": "initial"})
        display(pat_input)
        print("Token'i yazip hucreyi yeniden calistirin.")
    except ImportError:
        pass
    raise ValueError("GITHUB_PAT bulunamadi! Colab Secrets'a ekleyin.")

if os.path.exists(WORK_DIR):
    print("Repo zaten klonlanmis, guncelleniyor...")
    os.chdir(WORK_DIR)
    subprocess.run(["git", "fetch", "origin"], check=True, timeout=30, env=_clean_env)
    subprocess.run(["git", "reset", "--hard", "origin/main"], check=True, timeout=30, env=_clean_env)
    print("Repo guncellendi.")
else:
    auth_url = REPO_URL.replace("https://", f"https://{GITHUB_PAT}@")
    subprocess.run(["git", "clone", auth_url, WORK_DIR], check=True, timeout=60, env=_clean_env)
    os.chdir(WORK_DIR)
    print("Repo klonlandi.")

# Build script'inin var oldugunu dogrula
os.chdir(WORK_DIR + "/colab_docker")
assert os.path.exists("build_all.sh"), "build_all.sh bulunamadi!"
subprocess.run(["chmod", "+x", "build_all.sh"], check=True, env=_clean_env)
print(f"build_all.sh hazir. Commit: {subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True, env=_clean_env).stdout.strip()}")

## Hucre 4: Force-Rebuild Base + Video Modelleri

Video modelleri (codec fix iceren) zorunlu rebuild edilir.
Sirasiyla: base (altyapi), cogvideox, wan, ltx, hunyuan, svd, animatediff, wan25

> **Tahmini sure**: Base ~5dk, her video modeli ~15-30dk, toplam ~2 saat
> Colab baglantisi koparsa, bu hucreyi yeniden calistir (var olan imajlar atlanir)

In [ ]:
import os
_clean_env = os.environ.copy()
_clean_env.pop('LD_LIBRARY_PATH', None) # GLIBC hatasini onlemek icin
#@title Force-Rebuild: Base + Video Modelleri { display-mode: "form" }
import subprocess
import os
import time

os.chdir("/content/AI-Publisher/colab_docker")

VIDEO_MODELS = [
    "base",
    "cogvideox",
    "wan",
    "ltx",
    "hunyuan",
    "svd",
    "animatediff",
    "wan25",
]

print("Force-Rebuild basliyor: Video modelleri (codec fix ile)")
print(f"Modeller: {', '.join(VIDEO_MODELS)}")
print("=" * 60)

total_start = time.time()

# force_rebuild.sh ile calistir (Drive arsivlerini siler)
result = subprocess.run(
    ["bash", "force_rebuild.sh"] + VIDEO_MODELS,
    capture_output=False,
    text=True,
    timeout=72000  # 20 saat timeout
, env=_clean_env)

total_duration = (time.time() - total_start) / 60
print(f"\nVideo model build tamamlandi. Toplam sure: {total_duration:.1f} dk")
print(f"Cikis kodu: {result.returncode}")

## Hucre 5: Incremental Build - Ses/Efekt Modelleri

Ses ve efekt modelleri incremental build edilir. Drive'da varsa atlanir.
**Bagimsizdir** - video build'u beklemez, ayri oturumda calistirilabilir.

Modeller: xtts, audioldm2, wav2lip, musetalk, whisper, stablediffusion, kokorotts, f5tts, lora-trainer

In [ ]:
import os
_clean_env = os.environ.copy()
_clean_env.pop('LD_LIBRARY_PATH', None) # GLIBC hatasini onlemek icin
#@title Incremental Build: Ses/Efekt Modelleri { display-mode: "form" }
import subprocess
import os
import time

os.chdir("/content/AI-Publisher/colab_docker")

print("Incremental build basliyor: Ses/Efekt modelleri")
print("(Drive'da var olan imajlar atlanacak)")
print("=" * 60)

total_start = time.time()

result = subprocess.run(
    ["bash", "build_all.sh"],
    capture_output=False,
    text=True,
    timeout=43200  # 12 saat
, env=_clean_env)

total_duration = (time.time() - total_start) / 60
print(f"\nSes/Efekt model build tamamlandi. Toplam sure: {total_duration:.1f} dk")
print(f"Cikis kodu: {result.returncode}")

## Hucre 6: Imaj Dogrulama

Drive'daki imajlari kontrol et: boyut, tarih, sayi.
Video modellerinde codec fix'i dogrula (opsiyonel).

In [ ]:
#@title Imaj Dogrulama { display-mode: "form" }
import subprocess
import os
import json

DRIVE_DIR = "/content/drive/MyDrive/Colab Notebooks/docker/images"
MIN_SIZE = 100 * 1024 * 1024  # 100 MB

print("Drive'daki imajlar kontrol ediliyor...")
print("=" * 60)

if not os.path.exists(DRIVE_DIR):
    print(f"HATA: {DRIVE_DIR} bulunamadi! Drive mount edilmemis olabilir.")
else:
    import glob
    files = sorted(glob.glob(os.path.join(DRIVE_DIR, "*.tar.gz")))
    print(f"Toplam {len(files)} imaj dosyasi bulundu.\n")
    
    video_models = ["base", "cogvideox", "wan", "ltx", "hunyuan", "svd", "animatediff", "wan25"]
    audio_models = ["xtts", "audioldm2", "wav2lip", "musetalk", "whisper", "stablediffusion", "kokorotts", "f5tts", "lora-trainer"]
    
    # Model bazli durum
    all_models = video_models + audio_models
    drive_map = {os.path.basename(f).replace(".tar.gz", ""): f for f in files}
    
    for model in all_models:
        if model in drive_map:
            fpath = drive_map[model]
            size = os.path.getsize(fpath)
            size_mb = size / (1024 * 1024)
            mtime = os.path.getmtime(fpath)
            import datetime
            date_str = datetime.datetime.fromtimestamp(mtime).strftime("%Y-%m-%d %H:%M")
            status = "OK" if size >= MIN_SIZE else "KUCUK"
            print(f"  [{status:6s}] {model:20s} {size_mb:8.1f} MB  {date_str}")
        else:
            print(f"  [  EKSIK] {model:20s} Drive'da bulunamadi!")
    
    print()
    missing = [m for m in all_models if m not in drive_map]
    small = [m for m in all_models if m in drive_map and os.path.getsize(drive_map[m]) < MIN_SIZE]
    
    if not missing and not small:
        print("Tum imajlar basariyla build edilmis ve Drive'a kaydedilmis.")
    else:
        if missing:
            print(f"EKSIK: {', '.join(missing)}")
        if small:
            print(f"KUCUK/BOZUK: {', '.join(small)}")

print("\nDogrulama tamamlandi.")

## Hucre 7: Ozet ve Sonraki Adimlar

Build tamamlandiginda:
1. Imajlari yerel makineye indir: `colab_docker/pull_images.sh` (Drive'dan kopyala)
2. Docker compose ile baslat: `docker compose -f colab_docker/docker-compose.yml up -d`
3. Servisleri dogrula: `curl http://localhost:5001/health`

### Gerekli Portlar (localhost:5001-5016)
- 5001 cogvideox - Video (I2V)
- 5002 xtts - TTS (Turkce)
- 5003 audioldm2 - SFX
- 5004 wav2lip - Lip-Sync
- 5005 musetalk - Lip-Sync (alternatif)
- 5006 whisper - STT
- 5007 stablediffusion - Gorsel
- 5008 wan - Video (T2V)
- 5009 ltx - Video (I2V)
- 5010 hunyuan - Video
- 5011 kokorotts - TTS (hizli)
- 5012 svd - Video (I2V)
- 5013 animatediff - Video
- 5014 wan25 - Video (genis)
- 5015 f5tts - TTS
- 5016 lora-trainer - LoRA